In [1]:
import pandas as pd

In [2]:
dataset=pd.read_csv("Social_Network_Ads.csv")

In [3]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [4]:
dataset=dataset.drop("User ID",axis=1)

In [5]:
dataset

,Gender,Age,EstimatedSalary,Purchased
0,Male,19,19000,0
1,Male,35,20000,0
2,Female,26,43000,0
3,Female,27,57000,0
4,Male,19,76000,0
...,...,...,...,...
395,Female,46,41000,1
396,Male,51,23000,1
397,Female,50,20000,1
398,Male,36,33000,0


In [7]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [8]:
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,True
1,35,20000,0,True
2,26,43000,0,False
3,27,57000,0,False
4,19,76000,0,True
...,...,...,...,...
395,46,41000,1,False
396,51,23000,1,True
397,50,20000,1,False
398,36,33000,0,True


In [9]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [11]:
independent=dataset[['Age', 'EstimatedSalary','Gender_Male']]
dependent=dataset[['Purchased']]

In [13]:
# training set split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(independent,dependent,test_size=0.3,random_state=0)

In [24]:
# model selection 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
param_grid= {'solver':['newton-cg','lbfgs','liblinear','sag'],
             'penalty':['l2']}
grid=GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring="f1_weighted")
grid.fit(x_train,y_train)


Fitting 5 folds for each of 4 candidates, totalling 20 fits


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda3\Lib\site-packages\scipy\optimize\_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
C:\Anaconda3\Lib\site-packages\sklearn\utils\optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(


,estimator,LogisticRegression()
,param_grid,"{'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [26]:
# prediction 
grid_prediction = grid.predict(x_test)

In [33]:
# printing best parameter after tuning 
print(grid.best_params_)
re=grid.cv_results_

#confusion matrix 
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,grid_prediction)

# classification report 
from sklearn.metrics import classification_report
clf=classification_report(y_test,grid_prediction)


{'penalty': 'l2', 'solver': 'newton-cg'}


In [36]:
# evaluation 
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_prediction,average="weighted")
print("the best params{}".format(grid.best_params_),f1_macro)

the best params{'penalty': 'l2', 'solver': 'newton-cg'} 0.8906190214115365


In [37]:
print(cm)

[[74  5]
 [ 8 33]]


In [38]:
print(clf)

              precision    recall  f1-score   support

           0       0.90      0.94      0.92        79
           1       0.87      0.80      0.84        41

    accuracy                           0.89       120
   macro avg       0.89      0.87      0.88       120
weighted avg       0.89      0.89      0.89       120



In [43]:
from sklearn.metrics import roc_auc_score
auc_score=roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])
grid.predict_proba(x_test)[:,1]
print(auc_score)

0.9484408768138315


In [46]:
table=pd.DataFrame.from_dict(re)

In [47]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.791741,0.057836,0.028475,0.008375,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.835985,0.802399,0.644599,0.927778,0.909115,0.823975,0.100806,1
1,0.478411,0.049018,0.057140,0.007462,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.816207,0.775048,0.644599,0.927778,0.874356,0.807598,0.096542,2
2,0.176055,0.135633,0.111110,0.036906,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.503106,0.503106,0.591398,0.480769,0.520202,0.519716,0.037966,3
3,0.227857,0.072599,0.056332,0.013216,l2,sag,"{'penalty': 'l2', 'solver': 'sag'}",0.503106,0.503106,0.503106,0.480769,0.480769,0.494171,0.010943,4
